# Fig_2H-I
## DNA virus genomes: multivariate_GLM_partial_residuals

In [ ]:

## Multivariate quasibinomial GLM per viral GENOME and FAMILY (relative abundance,
## depth-independent), with each predictor shown as a partial residual, so the
## plot line equals the predictor's multivariate coefficient.
## Prediction set (consistent with the final PERMANOVA / diversity GLM): Meiofauna, O2_Cons and CH4_Flux
##
## Responses per-genome and per-family relative abundance (present in >=10% of
##   samples, to keep enough non-zero points to fit; this filter is ONLY for the per-taxon GLM).
##
## Outputs:
##   Table_multivariate_GLM_all.csv                 : every taxon x predictor test
##   Fig_partial_<level>_<taxon>__<predictor>.pdf   : one PDF per significant hit
##

In [17]:
library(vegan)
library(dplyr)
library(tidyr)
library(ggplot2)
library(car)

In [18]:
setwd("path/")

In [22]:
## CONFIG

VIRUS_FILE <- "data/DNA_Viruses119.txt"
MASTER     <- "data/Master_table_short.txt"
OUTDIR     <- "out_mvglm_DNA Virus"
PREV_MIN   <- 0.10
FDR_CUT    <- 0.055
PRED       <- c("Meiofauna", "O2_Cons","CH4_Flux")
TRT_ORDER  <- c("Unmanip","HMM","HM","LM")
TRT_COL    <- c(Unmanip="#00970080", HMM="#ffce04ff", HM="#2a579bcb", LM="#cd534cff")
## -----
dir.create(OUTDIR, showWarnings = FALSE)

In [20]:
## viral responses (relative abundance)

dv   <- read.table(VIRUS_FILE, header=TRUE, sep="\t", row.names=1, check.names=FALSE)
vmat <- as.matrix(dv)
fam  <- gsub("_like$","", gsub("DNA_([A-Za-z0-9]+)_.*","\\1", rownames(vmat)))
genome_rel <- decostand(t(vmat), "total")
family_rel <- decostand(t(rowsum(vmat, fam)), "total")

## measured predictors (depth-independent)
env <- read.table(MASTER, header=TRUE, sep="\t", check.names=FALSE,
                  na.strings=c("Na","NA",""), quote="", comment.char="")
colnames(env) <- c("Sample","Group","CH4_Conc","CH4_Flux","OM","DIC_Flux","O2_Cons","Macrofauna",
                   "Meiofauna")
env$Group <- factor(env$Group, levels = TRT_ORDER)


In [21]:
## core: multivariate quasibinomial GLM per taxon

fit_screen <- function(rel, level){
  rel <- rel[, colMeans(rel>0) >= PREV_MIN, drop=FALSE]
  mm  <- merge(data.frame(Sample=rownames(rel), rel, check.names=FALSE), env, by="Sample")
  mmz <- mm; mmz[PRED] <- lapply(mm[PRED], function(x) as.numeric(scale(x)))
  res <- list()
  for (tax in colnames(rel)){
    dd <- data.frame(y = mmz[[tax]], mmz[, PRED, drop=FALSE])
    dd <- dd[complete.cases(dd), ]
    if (nrow(dd) < length(PRED)+3) next
    m <- tryCatch(glm(y ~ ., data=dd, family=quasibinomial), error=function(e) NULL)
    if (is.null(m)) next
    s <- summary(m)$coefficients
    for (v in PRED) if (v %in% rownames(s))
      res[[paste(tax,v)]] <- data.frame(level=level, taxon=tax, predictor=v,
        estimate=round(s[v,1],3), p=s[v,4])
  }
  if (length(res)==0) return(NULL)
  r <- do.call(rbind, res); r$FDR <- p.adjust(r$p,"BH"); r$sig <- r$FDR < FDR_CUT; r
}

screens <- bind_rows(
  fit_screen(genome_rel, "Genome"),
  fit_screen(family_rel, "Family"))
write.csv(screens, file.path(OUTDIR,"Table_multivariate_GLM_all.csv"), row.names=FALSE)

cat("=== Multivariate GLM significant (FDR<", FDR_CUT, ") ===\n", sep="")
print(screens %>% group_by(level) %>% summarise(sig=sum(sig), tested=n(), .groups="drop") %>% as.data.frame())
cat("\nSignificant predictor frequency (GENOME):\n")
print(screens %>% filter(level=="Genome", sig) %>% count(predictor, sort=TRUE) %>% as.data.frame(), row.names=FALSE)


=== Multivariate GLM significant (FDR<0.055) ===
   level sig tested
1 Family   0     54
2 Genome  87    327

Significant predictor frequency (GENOME):
 predictor  n
 Meiofauna 31
   O2_Cons 30
  CH4_Flux 26


In [23]:
## Partial-residual plots: One per significant relationship

partial_resid_plots <- function(rel, level){
  sigset <- screens %>% filter(level==!!level, sig) %>% arrange(FDR)
  if (nrow(sigset)==0){ cat("No significant", level, "to plot.\n"); return(invisible()) }
  rel <- rel[, colMeans(rel>0) >= PREV_MIN, drop=FALSE]
  mm  <- merge(data.frame(Sample=rownames(rel), rel, check.names=FALSE), env, by="Sample")
  mmz <- mm; mmz[PRED] <- lapply(mm[PRED], function(x) as.numeric(scale(x)))
  for (i in seq_len(nrow(sigset))){
    tax <- sigset$taxon[i]; v <- sigset$predictor[i]
    dd <- data.frame(y=mmz[[tax]], mmz[, PRED, drop=FALSE], Group=mm$Group)
    dd <- dd[complete.cases(dd[,c("y",PRED)]),]
    m  <- glm(y ~ ., data=dd[,c("y",PRED)], family=quasibinomial)
      
    ## partial (component+residual) on the working/link scale:
    ## working residual + beta_v * x_v  -> line slope = the MULTIVARIATE coefficient
    pr <- residuals(m, type="working") + coef(m)[v]*dd[[v]]
    dfp <- data.frame(x=dd[[v]], pr=pr, Group=factor(dd$Group, levels=TRT_ORDER))
    p <- ggplot(dfp, aes(x, pr)) +
      geom_smooth(method="lm", se=TRUE, colour="#C0392B", fill="#F1948A", alpha=.25, linewidth=.8) +
      geom_point(aes(fill=Group), shape=21, size=3, colour="grey20", stroke=.4) +
      scale_fill_manual(values=TRT_COL) +
      labs(x=paste0(v," (z-scored)"), y=paste("Partial residual:", tax),
           title=paste0(tax, "  ~  ", v),
           subtitle=sprintf("multivariate GLM (measured predictors) | beta=%.2f, FDR=%.3f",
                            sigset$estimate[i], sigset$FDR[i])) +
      theme_bw(base_size=11)
    fname <- paste0("Fig_partial_", level, "_",
                    gsub("[^A-Za-z0-9]+","_", tax), "__", v, ".pdf")
    ggsave(file.path(OUTDIR, fname), p, width=6, height=5)
  }
  cat("Wrote", nrow(sigset), "individual", level, "partial-residual PDF(s).\n")
}
partial_resid_plots(genome_rel, "Genome")
partial_resid_plots(family_rel, "Family")

##  VIF (confirms the measured predictor set is not collinear)                               
mm <- merge(data.frame(Sample=rownames(genome_rel), genome_rel, check.names=FALSE), env, by="Sample")
vifmod <- lm(as.formula(paste0(make.names(colnames(genome_rel)[1])," ~ ", paste(PRED, collapse="+"))),
             data=setNames(mm, make.names(names(mm))))
cat("\n=== VIF (measured predictor set) ===\n"); print(round(vif(vifmod),2))
cat("\nOutputs -> ", OUTDIR, "/\n", sep="")

`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'
`geom_smooth()` using formula = 'y ~ x'


Wrote 87 individual Genome partial-residual PDF(s).
No significant Family to plot.

=== VIF (measured predictor set) ===
Meiofauna   O2_Cons  CH4_Flux 
     1.04      1.62      1.57 

Outputs -> out_mvglm_DNA Virus/
